# 01_fct_plays

DML: fct_plays — Fact: play events, deduplicated by play_id (SHA256).

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

src = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_play_history")
    .filter(F.col("ingestion_date") == ingestion_date)
    .filter(F.col("run_id") == run_id)
    .select("played_at", "track_id", "ingestion_date", "run_id")
    .withColumn(
        "play_id",
        F.sha2(F.concat_ws("||", F.col("played_at").cast("string"), F.col("track_id")), 256),
    )
    .dropDuplicates(["play_id"])
)

upsert_delta(src, f"{CATALOG}.{SILVER_SCHEMA}.fct_plays", ["play_id"])
print(f"fct_plays: {src.count()} rows upserted")